# Preparacion Snowflake Schema para Power BI

Este notebook toma el dataset final del proyecto (`Argenprop_limpio_con_indices.csv`) y genera una carpeta `snowflake/` con archivos CSV listos para cargar en Power BI.

Criterio usado: las variables categoricas con cardinalidad mayor a 2 se separan como dimensiones. Las variables numericas discretas, como ambientes, dormitorios, banos, conteos de servicios o cantidad de amenities, quedan en la tabla de hechos. Las excepciones numericas son `Comuna` y `Cluster`, porque funcionan como categorias de analisis.
Salida principal:
- `snowflake/fact_propiedades.csv`
- `snowflake/dim_*.csv`

El notebook calcula internamente perfil de cardinalidad, relaciones e integridad referencial, pero no exporta esos controles como CSV.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)

BASE_DIR = Path.cwd()
INPUT_CANDIDATES = [
    BASE_DIR / 'Argenprop_limpio_con_indices.csv',
    BASE_DIR / 'Argenprop_limpio.csv',
]
OUTPUT_DIR = BASE_DIR / 'snowflake'
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError('No se encontro Argenprop_limpio_con_indices.csv ni Argenprop_limpio.csv en el directorio del proyecto.')

df = pd.read_csv(INPUT_PATH)
print(f'Dataset cargado: {INPUT_PATH.name}')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]:,}')
df.head()

## 1. Perfilado de cardinalidad

Primero se calcula la cardinalidad de todas las columnas. Para el modelo snowflake orientado a Power BI se separan como dimensiones las variables categoricas con mas de dos valores. Las numericas discretas quedan en la fact table salvo `Comuna` y `Cluster`, que se tratan como categorias.

In [ ]:
def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    replacements = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ñ': 'n',
        'Á': 'a', 'É': 'e', 'Í': 'i', 'Ó': 'o', 'Ú': 'u', 'Ñ': 'n',
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r'[^a-z0-9]+', '_', value)
    value = re.sub(r'_+', '_', value).strip('_')
    return value or 'campo'

def clean_dimension_label(series: pd.Series) -> pd.Series:
    cleaned = series.astype('object').where(series.notna(), 'No disponible')
    cleaned = cleaned.astype(str).str.strip()
    cleaned = cleaned.mask(cleaned.eq('') | cleaned.str.lower().isin(['nan', 'none', 'nat']), 'No disponible')
    return cleaned

CLUSTER_NAMES = {
    '0': 'Barrios tradicionales accesibles',
    '1': 'Premium consolidado',
    '2': 'Residencial medio-alto',
    '3': 'Residencial accesible',
    '4': 'Alto valor por m2',
    '5': 'Compactos economicos',
}

CLUSTER_DESCRIPTIONS = {
    '0': 'Precios bajos, propiedades mas antiguas y valores de m2 relativamente bajos.',
    '1': 'Mayor precio promedio, mayor superficie, mas banos y precio por m2 muy elevado.',
    '2': 'Valores intermedios-altos, buena superficie y expensas elevadas.',
    '3': 'Menor cantidad de ambientes y banos, con precios relativamente bajos.',
    '4': 'Propiedades con pocos dormitorios pero alto precio por m2 y precios elevados.',
    '5': 'Departamentos mas chicos, con menor superficie y precios bajos/intermedios.',
}

def is_dimension_candidate(series: pd.Series, column: str, min_cardinality: int = 3) -> bool:
    nunique = series.nunique(dropna=True)
    if nunique < min_cardinality:
        return False
    lowered = column.lower()
    excluded_tokens = ['id_registro', 'link', 'latitud', 'longitud', 'precio_m2']
    if any(token in lowered for token in excluded_tokens):
        return False
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series) or isinstance(series.dtype, pd.CategoricalDtype):
        return True
    return False

profile = pd.DataFrame({
    'columna': df.columns,
    'tipo': [str(df[c].dtype) for c in df.columns],
    'cardinalidad': [df[c].nunique(dropna=True) for c in df.columns],
    'nulos': [df[c].isna().sum() for c in df.columns],
    'pct_nulos': [round(df[c].isna().mean() * 100, 2) for c in df.columns],
})
profile['candidata_dimension'] = [is_dimension_candidate(df[c], c) for c in df.columns]
profile.sort_values(['candidata_dimension', 'cardinalidad'], ascending=[False, False])

## 2. Seleccion de dimensiones

El notebook detecta candidatas automaticamente y fuerza algunas dimensiones importantes para el analisis inmobiliario. La seleccion actual deja como dimensiones: estado, disposicion, tipo de unidad, barrio, comuna, cluster, subte cercano, linea de subte, hospital cercano y avenida cercana. Calles, alturas, links, pisos y variables numericas discretas quedan directamente en `fact_propiedades`.

In [ ]:
AUTO_DIMENSIONS = profile.loc[profile['candidata_dimension'], 'columna'].tolist()

FORCED_DIMENSION_PATTERNS = [
    'estado', 'comuna', 'barrio', 'cluster',
    'disposicion', 'tipo_unidad', 'linea_subte', 'subte_cercano',
    'hospital_cercano', 'avenida_cercana'
]
NUMERIC_DIMENSION_EXCEPTIONS = ['comuna', 'cluster']

forced_dimensions = []
for column in df.columns:
    normalized = normalize_name(column)
    is_numeric = pd.api.types.is_numeric_dtype(df[column])
    is_allowed_numeric_dimension = any(pattern in normalized for pattern in NUMERIC_DIMENSION_EXCEPTIONS)
    if df[column].nunique(dropna=True) > 2 and any(pattern in normalized for pattern in FORCED_DIMENSION_PATTERNS) and (not is_numeric or is_allowed_numeric_dimension):
        forced_dimensions.append(column)

DIMENSION_COLUMNS = sorted(set(AUTO_DIMENSIONS + forced_dimensions), key=list(df.columns).index)

# Campos de mucha granularidad que conviene conservar en la fact table como atributos descriptivos.
KEEP_AS_FACT_ATTRIBUTES = [c for c in ['original_Calle', 'original_Altura', 'original_Link', 'imputada_Piso'] if c in df.columns]
DIMENSION_COLUMNS = [c for c in DIMENSION_COLUMNS if c not in KEEP_AS_FACT_ATTRIBUTES]

dimension_summary = profile[profile['columna'].isin(DIMENSION_COLUMNS)].copy()
dimension_summary = dimension_summary.sort_values('cardinalidad', ascending=False)
print('Dimensiones seleccionadas:', len(DIMENSION_COLUMNS))
dimension_summary[['columna', 'tipo', 'cardinalidad', 'nulos']]

## 3. Construccion de dimensiones y tabla de hechos

Cada dimension recibe una llave sustituta (`*_key`) y una columna descriptiva (`*_valor`). La excepcion es `dim_cluster`, que expone `cluster_numero`, `cluster_nombre`, `cluster_etiqueta_powerbi` y `cluster_descripcion` para que sea mas comoda de usar en Power BI. La tabla de hechos conserva metricas, flags binarios, identificadores, atributos directos y las llaves hacia las dimensiones.

In [ ]:
fact = df.copy()
dimensions = {}
dimension_lookups = {}
relationships = []

id_column = 'sintetica_id_registro' if 'sintetica_id_registro' in fact.columns else None
if id_column is None:
    id_column = 'sintetica_id_registro'
    fact.insert(0, id_column, np.arange(1, len(fact) + 1))

used_table_names = set()

for column in DIMENSION_COLUMNS:
    base_name = normalize_name(column)
    for prefix in ['imputada_', 'enriquecida_', 'sintetica_', 'original_']:
        if base_name.startswith(prefix):
            base_name = base_name[len(prefix):]
    table_name = f'dim_{base_name}'
    original_table_name = table_name
    suffix = 2
    while table_name in used_table_names:
        table_name = f'{original_table_name}_{suffix}'
        suffix += 1
    used_table_names.add(table_name)

    key_col = f'{base_name}_key'
    value_col = f'{base_name}_valor'
    helper_col = f'__{base_name}_valor_limpio'

    fact[helper_col] = clean_dimension_label(fact[column])
    dimension_values = fact[helper_col].drop_duplicates().tolist()
    if table_name == 'dim_comuna':
        dimension_values = sorted(dimension_values, key=lambda x: pd.to_numeric(x, errors='coerce'))
    else:
        dimension_values = sorted(dimension_values, key=lambda x: str(x))
    dim = pd.DataFrame({value_col: dimension_values})
    dim.insert(0, key_col, np.arange(1, len(dim) + 1))

    if table_name == 'dim_cluster':
        dim['cluster_numero'] = pd.to_numeric(dim[value_col], errors='coerce').astype('Int64')
        dim['cluster_nombre'] = dim[value_col].map(CLUSTER_NAMES).fillna('Cluster sin clasificar')
        dim['cluster_etiqueta_powerbi'] = 'Cluster ' + dim[value_col].astype(str) + ' - ' + dim['cluster_nombre']
        dim['cluster_descripcion'] = dim[value_col].map(CLUSTER_DESCRIPTIONS).fillna('Sin descripcion disponible.')
    elif table_name == 'dim_comuna':
        dim['comuna_numero'] = pd.to_numeric(dim[value_col], errors='coerce').astype('Int64')
        dim['comuna_nombre'] = 'Comuna ' + dim['comuna_numero'].astype(str)

    dim_lookup = dim[[key_col, value_col]].copy()
    dimension_lookups[table_name] = dim_lookup.copy()
    fact = fact.merge(dim_lookup, left_on=helper_col, right_on=value_col, how='left')
    fact = fact.drop(columns=[column, helper_col, value_col])
    if table_name == 'dim_cluster':
        dim = dim.drop(columns=[value_col])
    elif table_name == 'dim_comuna':
        dim = dim.drop(columns=[value_col, 'comuna_numero'])

    dimensions[table_name] = dim
    relationships.append({
        'tabla_hechos': 'fact_propiedades',
        'columna_hechos': key_col,
        'tabla_dimension': table_name,
        'columna_dimension': key_col,
        'campo_original': column,
        'cardinalidad_dimension': len(dim),
    })

# Orden recomendado: id, llaves, atributos descriptivos, metricas/flags restantes.
key_columns = [r['columna_hechos'] for r in relationships]
remaining_columns = [c for c in fact.columns if c not in [id_column] + key_columns]
fact = fact[[id_column] + key_columns + remaining_columns]

print(f'Tabla de hechos: {fact.shape[0]:,} filas x {fact.shape[1]:,} columnas')
print(f'Dimensiones creadas: {len(dimensions):,}')
pd.DataFrame(relationships).head(30)

## 4. Normalizacion tipo snowflake para ubicacion

Para que el modelo sea un snowflake schema real, se normaliza la jerarquia geografica: `Barrio` apunta a `Comuna`. En Power BI la relacion correcta queda como `fact_propiedades -> dim_barrio -> dim_comuna`. La tabla de hechos no conserva `comuna_key` directo, porque la comuna se obtiene desde el barrio.

In [ ]:
def find_dimension_table(token: str):
    matches = [name for name in dimensions if token in name]
    return matches[0] if matches else None

barrio_table = find_dimension_table('barrio')
comuna_table = find_dimension_table('comuna')

snowflake_relationships = []
if barrio_table and comuna_table and 'enriquecida_Barrio' in df.columns and 'enriquecida_Comuna' in df.columns:
    barrio_dim = dimensions[barrio_table].copy()
    comuna_dim = dimensions[comuna_table].copy()
    comuna_lookup = dimension_lookups[comuna_table].copy()
    barrio_value_col = [c for c in barrio_dim.columns if c.endswith('_valor')][0]
    comuna_key_col = [c for c in comuna_dim.columns if c.endswith('_key')][0]
    comuna_value_col = [c for c in comuna_lookup.columns if c.endswith('_valor')][0]

    barrio_comuna_map = (
        df[['enriquecida_Barrio', 'enriquecida_Comuna']]
        .assign(
            enriquecida_Barrio=lambda x: clean_dimension_label(x['enriquecida_Barrio']),
            enriquecida_Comuna=lambda x: clean_dimension_label(x['enriquecida_Comuna']),
        )
        .drop_duplicates()
    )
    barrio_comuna_map = barrio_comuna_map.merge(
        comuna_lookup,
        left_on='enriquecida_Comuna',
        right_on=comuna_value_col,
        how='left',
    )

    barrio_dim = barrio_dim.merge(
        barrio_comuna_map[['enriquecida_Barrio', comuna_key_col]],
        left_on=barrio_value_col,
        right_on='enriquecida_Barrio',
        how='left',
    ).drop(columns=['enriquecida_Barrio'])
    dimensions[barrio_table] = barrio_dim
    snowflake_relationships.append({
        'tabla_desde': barrio_table,
        'columna_desde': comuna_key_col,
        'tabla_hacia': comuna_table,
        'columna_hacia': comuna_key_col,
        'descripcion': 'Jerarquia geografica Barrio -> Comuna',
    })

    # Snowflake estricto: la fact no se relaciona directo con Comuna; llega via Barrio.
    if comuna_key_col in fact.columns:
        fact = fact.drop(columns=[comuna_key_col])
    relationships = [rel for rel in relationships if rel['tabla_dimension'] != comuna_table]
    key_columns = [r['columna_hechos'] for r in relationships]

pd.DataFrame(snowflake_relationships)

## 5. Exportacion de CSV para Power BI

Todos los CSV de tablas se guardan dentro de `snowflake/`. Antes de exportar se eliminan los CSV anteriores de esa carpeta para que no queden dimensiones viejas. Se usa `utf-8-sig` para que Power BI reconozca correctamente los encabezados y caracteres especiales. Los perfiles, relaciones y validaciones se calculan internamente, pero no se exportan como archivos.

In [ ]:
for old_csv in OUTPUT_DIR.glob('*.csv'):
    old_csv.unlink()

fact_path = OUTPUT_DIR / 'fact_propiedades.csv'
fact.to_csv(fact_path, index=False, encoding='utf-8-sig')

manifest_rows = [{
    'archivo': fact_path.name,
    'tabla': 'fact_propiedades',
    'tipo': 'hechos',
    'filas': len(fact),
    'columnas': fact.shape[1],
}]

for table_name, dim in dimensions.items():
    path = OUTPUT_DIR / f'{table_name}.csv'
    dim.to_csv(path, index=False, encoding='utf-8-sig')
    manifest_rows.append({
        'archivo': path.name,
        'tabla': table_name,
        'tipo': 'dimension',
        'filas': len(dim),
        'columnas': dim.shape[1],
    })

relationships_df = pd.DataFrame(relationships)
snowflake_relationships_df = pd.DataFrame(snowflake_relationships)
manifest_df = pd.DataFrame(manifest_rows).sort_values(['tipo', 'tabla'])

print(f'CSV exportados en: {OUTPUT_DIR.resolve()}')
manifest_df

## 6. Chequeos finales

Estos chequeos verifican que no se hayan perdido filas, que las llaves de dimensiones no tengan nulos y que cada relacion pueda cargarse en Power BI sin problemas basicos de integridad referencial.

In [ ]:
assert len(fact) == len(df), 'La tabla de hechos no conserva la cantidad original de filas.'

key_columns_with_nulls = fact[key_columns].isna().sum()
key_columns_with_nulls = key_columns_with_nulls[key_columns_with_nulls > 0]
assert key_columns_with_nulls.empty, f'Hay llaves nulas en fact_propiedades: {key_columns_with_nulls.to_dict()}'

integrity_results = []
for rel in relationships:
    fact_values = set(fact[rel['columna_hechos']].dropna().astype(int).unique())
    dim_values = set(dimensions[rel['tabla_dimension']][rel['columna_dimension']].dropna().astype(int).unique())
    missing = fact_values - dim_values
    integrity_results.append({
        'tabla_dimension': rel['tabla_dimension'],
        'columna_hechos': rel['columna_hechos'],
        'ok': len(missing) == 0,
        'claves_faltantes': sorted(missing)[:10],
    })

integrity_df = pd.DataFrame(integrity_results)
print('Filas conservadas:', len(fact))
print('Relaciones validadas:', integrity_df['ok'].sum(), '/', len(integrity_df))
integrity_df